# HANDWRITTEN DIGITS RECOGNITION

## 1. PACKAGES

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import helper_utils

## 2. DATA PIPELINE

In [5]:
# Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # two numbers are represented for mean and standard deviation for the whole dataset
])

### Load MNIST Dataset

In [6]:
#load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download = True)

In [ ]:
# Get the first sample (index 0), as a (image, label) tuple
image_pil, label = train_dataset_without_transform[0] # Get the first image

print(f"Image type:        {type(image_pil)}")
# Since `image_pil` is a PIL Image object, its dimensions are accessed using the .size attribute.
print(f"Image Dimensions:  {image_pil.size}")
print(f"Label Type:        {type(label)}")
print(f"Label value:       {label}")

### Create data loaders

In [7]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

## 3. Building a neural network

In [8]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.Flatten = nn.Flatten() #make the 2D image to 1D vector
        self.layers = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.Flatten(x)
        x = self.layers(x)
        return x

## 4. Device and Optimizer

In [13]:
# Check for GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

# Initialize model and move to device
model = MNISTClassifier().to(device)

# Loss function and Optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: MPS (Apple Silicon GPU)


In [10]:
def train_epoch(model, train_loader, loss_function, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        # Track progress
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        #print every 100 batches
        if batch_idx % 100 == 0 and batch_idx > 0:
            avg_loss = running_loss / 100
            accuracy = 100. * correct / total
            print(f'  [{batch_idx * 64}/{60000}]'
                  f'Loss: {avg_loss:.3f} | Accuracy: {accuracy:.1f}%')
            running_loss = 0.0

### Evaluation

In [11]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return 100. * correct / total

## Put all together

In [12]:
# Training loop
num_epochs = 10
for epoch in range (num_epochs):
    print(f'\nEpoch {epoch + 1}')
    train_epoch(model, train_loader, loss_function, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f'Test Accuracy: {accuracy:.2f}%')


Epoch 1
  [6400/60000]Loss: 0.640 | Accuracy: 81.4%
  [12800/60000]Loss: 0.315 | Accuracy: 86.1%
  [19200/60000]Loss: 0.281 | Accuracy: 88.0%
  [25600/60000]Loss: 0.239 | Accuracy: 89.2%
  [32000/60000]Loss: 0.202 | Accuracy: 90.2%
  [38400/60000]Loss: 0.202 | Accuracy: 90.8%
  [44800/60000]Loss: 0.164 | Accuracy: 91.4%
  [51200/60000]Loss: 0.166 | Accuracy: 91.9%
  [57600/60000]Loss: 0.160 | Accuracy: 92.2%
Test Accuracy: 95.57%

Epoch 2
  [6400/60000]Loss: 0.145 | Accuracy: 96.0%
  [12800/60000]Loss: 0.126 | Accuracy: 96.2%
  [19200/60000]Loss: 0.123 | Accuracy: 96.3%
  [25600/60000]Loss: 0.104 | Accuracy: 96.4%
  [32000/60000]Loss: 0.108 | Accuracy: 96.5%
  [38400/60000]Loss: 0.111 | Accuracy: 96.5%
  [44800/60000]Loss: 0.104 | Accuracy: 96.6%
  [51200/60000]Loss: 0.105 | Accuracy: 96.6%
  [57600/60000]Loss: 0.094 | Accuracy: 96.7%
Test Accuracy: 97.10%

Epoch 3
  [6400/60000]Loss: 0.076 | Accuracy: 97.8%
  [12800/60000]Loss: 0.078 | Accuracy: 97.6%
  [19200/60000]Loss: 0.083 | Acc